# 09 - Multi-Core Synchronization

**Objective:** Use multiple tProc cores simultaneously, manage triggers, and handle cross-core dependencies. This notebook covers:
- Basic multi-core configuration
- Trigger-based synchronization between cores
- Cross-core wait operations
- Coordinated multi-pulse sequences

## 1. Setup

In [ ]:
# Standard imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import logging
import time

from qick import *
from qick.asm_v2 import AveragerProgramV2, AsmV2

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

# Connect to the board (adjust the path to your firmware)
BITSTREAM_PATH = '/path/to/your/firmware.bit'  # <--- CHANGE THIS
soc = QickSoc(BITSTREAM_PATH)
soccfg = soc

# Define hardware channels
GEN_CH_0 = 0      # Generator channel for core 0
GEN_CH_1 = 1      # Generator channel for core 1
RO_CH = 0         # Readout channel

print(f"Available tProc cores: {soc.num_tprocs}")
print(f"Firmware version: {soc.get_cfg()['fw_version']}")

## 2. Why Multi-Core?

The QICK tProc v2 has multiple independent processing cores. Multi-core programming allows you to:
- Run independent sequences in parallel
- Synchronize operations across different DAC channels
- Implement complex feedback loops with dedicated cores
- Reduce sequence latency by distributing tasks

Each core has its own program memory and instruction pointer, but they share access to DAC and ADC resources.

## 3. Basic Multi-Core Configuration

The simplest multi-core setup runs independent sequences on different cores. Each core can control different DAC channels.

In [ ]:
class BasicMultiCoreProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        # Declare resources for each core
        # Note: Different cores can use different channels
        self.declare_gen(ch=cfg['gen_ch_0'], nqz=1, core=0)
        self.declare_gen(ch=cfg['gen_ch_1'], nqz=1, core=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Define pulses for core 0
        self.add_pulse(
            ch=cfg['gen_ch_0'], name="pulse_core0",
            style="const",
            freq=cfg['freq'], length=cfg['pulse_len'],
            phase=0, gain=cfg['gain']
        )

        # Define pulses for core 1
        self.add_pulse(
            ch=cfg['gen_ch_1'], name="pulse_core1",
            style="const",
            freq=cfg['freq'], length=cfg['pulse_len'],
            phase=90, gain=cfg['gain']
        )

        # Configure readout (on core 0 by default)
        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro", 
                               freq=cfg['freq'], gen_ch=cfg['gen_ch_0'])

    def _body(self, cfg):
        # This body runs on core 0
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)
        self.pulse(ch=cfg['gen_ch_0'], name="pulse_core0", t=0)

    def _body_core1(self, cfg):
        # This body runs on core 1 (must be defined separately)
        self.pulse(ch=cfg['gen_ch_1'], name="pulse_core1", t=0)

config_multi = {
    'gen_ch_0': GEN_CH_0,
    'gen_ch_1': GEN_CH_1,
    'ro_ch': RO_CH,
    'freq': 100.0,
    'pulse_len': 0.2,
    'gain': 0.8,
    'trig_time': 0.1,
    'ro_len': 0.5
}

# Create program
prog = BasicMultiCoreProgram(soccfg, reps=1, final_delay=0.5, cfg=config_multi)

# Access the assembly for each core
print("=== Core 0 Assembly ===")
print(prog.asm(core=0))
print("\n=== Core 1 Assembly ===")
print(prog.asm(core=1))

**Note:** When running multi-core programs, you need to load configurations to each core separately and start them simultaneously.

In [ ]:
# Alternative: Direct multi-core execution without the program class
# This approach gives you more control over each core's program

def run_basic_multi_core():
    """Run independent sequences on two cores."""
    # Configure core 0
    config0 = QickConfig(core=0)
    config0.pulse(ch=GEN_CH_0, style="const", freq=100e6, length=0.2, gain=0.8)
    config0.pulse(ch=GEN_CH_0, style="const", length=0.2, gain=0)  # Off pulse
    
    # Configure core 1
    config1 = QickConfig(core=1)
    config1.pulse(ch=GEN_CH_1, style="const", freq=100e6, length=0.2, gain=0.5, phase=90)
    config1.pulse(ch=GEN_CH_1, style="const", length=0.2, gain=0)
    
    # Load and start
    soc.load_config(config0, core=0)
    soc.load_config(config1, core=1)
    
    # Start both cores simultaneously
    start_time = time.time()
    soc.start_core(0)
    soc.start_core(1)
    
    # Wait for completion
    soc.wait_core(0)
    soc.wait_core(1)
    elapsed = time.time() - start_time
    
    print(f"Both cores completed in {elapsed*1000:.2f} ms")

# Uncomment to run:
# run_basic_multi_core()

## 4. Trigger-Based Synchronization

Triggers allow cores to communicate with each other in real-time. A core can send a trigger pulse that another core can wait for. This enables precise coordination.

In [ ]:
class TriggerSyncProgram(AveragerProgramV2):
    """Core 0 triggers Core 1 to start a sequence."""
    
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch_0'], nqz=1, core=0)
        self.declare_gen(ch=cfg['gen_ch_1'], nqz=1, core=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Define pulses
        self.add_pulse(ch=cfg['gen_ch_0'], name="pulse0",
                       style="const", freq=cfg['freq'], 
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain'])
        
        self.add_pulse(ch=cfg['gen_ch_1'], name="pulse1",
                       style="const", freq=cfg['freq'], 
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain'])

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq'], gen_ch=cfg['gen_ch_0'])

    def _body(self, cfg):
        """Core 0: Send trigger then play pulse."""
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)
        
        # Send trigger to core 1 at t=0
        self.trigger(pins=[1], t=0, width=0.02, dest=1)  # dest=1 sends to core 1
        
        # Core 0 plays its pulse
        self.pulse(ch=cfg['gen_ch_0'], name="pulse0", t=0.1)

    def _body_core1(self, cfg):
        """Core 1: Wait for trigger then play pulse."""
        # Wait for trigger from core 0 (timeout 1 ms)
        self.wait_trigger(t=0, timeout=1000)
        
        # Play pulse immediately after receiving trigger
        self.pulse(ch=cfg['gen_ch_1'], name="pulse1", t=0)

config_trigger = {
    'gen_ch_0': GEN_CH_0,
    'gen_ch_1': GEN_CH_1,
    'ro_ch': RO_CH,
    'freq': 100.0,
    'pulse_len': 0.15,
    'gain': 0.7,
    'trig_time': 0.05,
    'ro_len': 0.5
}

prog = TriggerSyncProgram(soccfg, reps=1, final_delay=0.5, cfg=config_trigger)
print("=== Trigger Synchronization Assembly ===")
print("Core 0:")
print(prog.asm(core=0))
print("\nCore 1:")
print(prog.asm(core=1))

**How trigger synchronization works:**
- `trigger(pins=[1], t=0, dest=1)` sends a trigger pulse to core 1
- `wait_trigger(t=0, timeout=1000)` on core 1 pauses execution until a trigger arrives
- The timeout prevents deadlock if the trigger never arrives
- This creates a master-slave relationship between cores

## 5. Cross-Core Wait Operations

The `wait_core` instruction allows one core to wait for another core to complete its current operation. This is useful for sequential dependencies.

In [ ]:
class CrossCoreWaitProgram(AveragerProgramV2):
    """Core 0 waits for Core 1 to finish before continuing."""
    
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch_0'], nqz=1, core=0)
        self.declare_gen(ch=cfg['gen_ch_1'], nqz=1, core=1)

        # Define a long pulse for core 1
        self.add_pulse(ch=cfg['gen_ch_1'], name="long_pulse",
                       style="const", freq=cfg['freq'], 
                       length=cfg['long_len'], phase=0, gain=cfg['gain'])
        
        # Define a short pulse for core 0
        self.add_pulse(ch=cfg['gen_ch_0'], name="short_pulse",
                       style="const", freq=cfg['freq'], 
                       length=cfg['short_len'], phase=0, gain=cfg['gain'])

    def _body(self, cfg):
        """Core 0: Wait for core 1, then play pulse."""
        # Wait for core 1 to finish
        # This is a placeholder - actual implementation varies by QICK version
        # self.wait_core(core=1, t=0)
        
        # Play short pulse
        self.pulse(ch=cfg['gen_ch_0'], name="short_pulse", t=0)

    def _body_core1(self, cfg):
        """Core 1: Play long pulse (this takes time)."""
        self.pulse(ch=cfg['gen_ch_1'], name="long_pulse", t=0)

config_wait = {
    'gen_ch_0': GEN_CH_0,
    'gen_ch_1': GEN_CH_1,
    'freq': 100.0,
    'long_len': 0.5,
    'short_len': 0.1,
    'gain': 0.5
}

print("Note: wait_core functionality may require specific firmware support.")
print("Check your QICK documentation for the exact syntax for cross-core waits.")

## 6. Practical Example: Simultaneous Readout and Control

A common multi-core use case is running a qubit control sequence on one core while performing readout on another.

In [ ]:
class ControlAndReadoutProgram(AveragerProgramV2):
    """
    Core 0: Qubit control sequence (pulse, wait, pulse)
    Core 1: Readout sequence triggered at specific times
    """
    
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['control_ch'], nqz=1, core=0)
        self.declare_gen(ch=cfg['readout_ch'], nqz=1, core=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Control pulses (core 0)
        self.add_pulse(ch=cfg['control_ch'], name="pi_pulse",
                       style="gaussian", freq=cfg['qubit_freq'],
                       length=cfg['pi_len'], phase=0, gain=cfg['control_gain'])
        
        # Readout pulse (core 1)
        ramp_len = 0.05
        self.add_gauss(ch=cfg['readout_ch'], name="ramp",
                       sigma=ramp_len/8, length=ramp_len, even_length=True)
        self.add_pulse(ch=cfg['readout_ch'], name="readout_pulse",
                       style="flat_top", envelope="ramp",
                       freq=cfg['readout_freq'], length=cfg['ro_len'],
                       phase=0, gain=cfg['readout_gain'])

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['readout_freq'], gen_ch=cfg['readout_ch'])

    def _body(self, cfg):
        """Core 0: Control sequence."""
        # Play π pulse
        self.pulse(ch=cfg['control_ch'], name="pi_pulse", t=0, tag='pi')
        
        # Wait for qubit evolution (Rabi or Ramsey)
        self.delay(cfg['evolution_time'])
        
        # Play second π/2 pulse (for Ramsey)
        self.pulse(ch=cfg['control_ch'], name="pi_pulse", t=cfg['evolution_time'] + 0.1, tag='pi2')

    def _body_core1(self, cfg):
        """Core 1: Readout at specific time."""
        # Wait for control sequence to finish
        self.delay(cfg['evolution_time'] + 0.2)
        
        # Trigger readout
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0, ddr4=True)
        self.pulse(ch=cfg['readout_ch'], name="readout_pulse", t=0)

config_cr = {
    'control_ch': GEN_CH_0,
    'readout_ch': GEN_CH_1,
    'ro_ch': RO_CH,
    'qubit_freq': 100.0,
    'readout_freq': 95.0,
    'pi_len': 0.1,
    'evolution_time': 0.5,
    'control_gain': 0.8,
    'readout_gain': 0.5,
    'ro_len': 0.8
}

print("Control and Readout Program Structure:")
print(f"- Core 0: π pulse at t=0, evolution for {config_cr['evolution_time']} µs, then π/2 pulse")
print(f"- Core 1: Readout pulse at t={config_cr['evolution_time'] + 0.2} µs")
print(f"- Readout frequency: {config_cr['readout_freq']} MHz (detuned from qubit)")

# Uncomment to run:
# prog = ControlAndReadoutProgram(soccfg, reps=100, final_delay=1.0, cfg=config_cr)
# iq_data = prog.acquire_decimated(soc, rounds=100)
# Use this data to measure Rabi oscillations or Ramsey fringes

## 7. Debugging Multi-Core Programs

Debugging multi-core programs can be challenging. Here are some tips:

In [ ]:
def debug_multi_core():
    """Helper functions for debugging multi-core execution."""
    
    # Check core status
    for core in range(soc.num_tprocs):
        status = soc.get_core_status(core)
        print(f"Core {core} status: {status}")
        # Status bits: 1=running, 2=waiting, 4=finished, 8=error
    
    # Get core program counter
    for core in range(soc.num_tprocs):
        pc = soc.get_core_pc(core)
        print(f"Core {core} PC: {pc}")
    
    # Reset all cores
    soc.reset_cores()
    print("All cores reset")

# Common issues and solutions:
# 1. Deadlock: One core waiting for a trigger that never arrives → increase timeout
# 2. Resource conflict: Two cores trying to use the same DAC/ADC → assign unique channels
# 3. Timing mismatch: Triggers arriving at wrong times → verify trigger timing alignment
# 4. Program memory overflow: Each core has limited program memory → reduce instruction count

print("Debugging tips for multi-core programs:")
print("1. Run cores individually first to verify each program works")
print("2. Use shorter sequences when debugging")
print("3. Add print statements in the Python code, not in the tProc assembly")
print("4. Monitor core status registers during execution")

## 8. Summary

You have learned:
- How to configure and run multiple tProc cores
- How to synchronize cores using triggers and wait operations
- How to implement parallel control and readout sequences
- Debugging strategies for multi-core programs

**Key takeaways:**
- Multi-core programming enables parallel and coordinated operations
- Triggers are the primary mechanism for core-to-core communication
- Each core has independent program memory and resources
- Always test cores individually before combining them

**Next steps:** Proceed to [`10_Streaming_And_RealTime_Processing.ipynb`](./10_Streaming_And_RealTime_Processing.ipynb) to learn about IQ streaming and on-FPGA processing.